In [1]:
import os
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from typing import Dict, List, Tuple, Optional
import pandas as pd
import numpy as np

In [3]:
USER_SPARSE = ["121", "122", "124", "125", "126", "127", "128", "129"]
USER_DENSE  = []
ITEM_SPARSE = ["206", "207", "210", "216"]
ITEM_DENSE  = []

vocabulary_size = {
        '101': 238635, '121': 98,     '122': 14,     '124': 3,
        '125': 8,      '126': 4,      '127': 4,      '128': 3,
        '129': 5,      '206': 7092,   '207': 285825, '210': 82412,
        '216': 113262, '508': 6057,   '509': 118936, '702': 57234,
        '853': 40002,  '301': 3, }

In [4]:
data_path = "/work/home/maben/project/rec_sys/projects/Ali_CCP/datasets/datasetsali_ccp_test.csv"
df = pd.read_csv(data_path)

In [5]:
df.head()

,click,purchase,101,121,122,124,125,126,127,128,...,127_14,150_14,D109_14,D110_14,D127_14,D150_14,D508,D509,D702,D853
0,0,0,0,1,6,1,4,0,1,1,...,2214,1799,0.3667,0.09235,0.0856,0.12730,0.0000,0.000,0.0,0.00000
1,1,0,114955,1,5,1,2,0,3,1,...,8343,71,0.1323,0.27700,0.0856,0.51800,0.4307,0.000,0.0,0.13060
2,0,0,7619,1,5,1,2,0,2,1,...,6,101,0.1938,0.09235,0.2212,0.31300,0.1953,0.000,0.0,0.00000
3,0,0,0,22,8,1,5,1,1,1,...,5373,28,0.0835,0.27700,0.0856,0.04523,0.4550,0.179,0.0,0.10974
4,0,0,65686,1,6,1,4,0,1,1,...,218,44,0.1323,0.09235,0.1986,0.15300,0.5120,0.000,0.0,0.27540


In [6]:
df.columns

Index(['click', 'purchase', '101', '121', '122', '124', '125', '126', '127',
       '128', '129', '205', '206', '207', '210', '216', '508', '509', '702',
       '853', '301', '109_14', '110_14', '127_14', '150_14', 'D109_14',
       'D110_14', 'D127_14', 'D150_14', 'D508', 'D509', 'D702', 'D853'],
      dtype='object')

In [17]:
from typing import List, Dict, Optional, Union
from dataclasses import dataclass
import torch
import torch.nn as nn


@dataclass
class SparseFeature:
    """稀疏特征（类别特征）"""
    name: str
    vocab_size: int
    embed_dim: int
    shared_embed: Optional[str] = None  # 共享嵌入的特征名


@dataclass
class DenseFeature:
    """稠密特征（数值特征）"""
    name: str
    dim: int = 1  # 特征维度


@dataclass
class SequenceFeature:
    """序列特征（用户行为序列）"""
    name: str
    vocab_size: int
    embed_dim: int
    max_len: int
    pooling: str = 'mean'  # 'mean', 'sum', 'max', 'attention'
    shared_embed: Optional[str] = None

In [7]:
def build_recall_tables(
    df: pd.DataFrame,
    user_col: str = 'user_id',
    item_col: str = 'item_id',
    label_col: str = 'click',
    hard_neg_col: Optional[str] = None,
) -> Dict:
    """
    从原始交互表提取召回训练所需的各类数据结构。

    Returns:
        pos_df       - 正样本表 (click==1)
        hard_neg_df  - 困难负样本表 (click==0，或由 hard_neg_col 指定)
        item_pool    - np.ndarray，全量物品ID
        popularity   - dict {item_id: float}，归一化热度
        user_history - dict {user_id: set}，用户交互过的物品集合
    """
    pos_df = df[df[label_col] == 1].reset_index(drop=True)

    if hard_neg_col is not None:
        hard_neg_df = df[[user_col, hard_neg_col]].rename(columns={hard_neg_col: item_col}).dropna().reset_index(drop=True)
    else:
        hard_neg_df = df[df[label_col] == 0].reset_index(drop=True)

    item_pool = df[item_col].unique()

    counts = df[df[label_col] == 1][item_col].value_counts()
    total = counts.sum()
    popularity = {item: cnt / total for item, cnt in counts.items()}

    user_history = df[df[label_col] == 1].groupby(user_col)[item_col].apply(set).to_dict()

    return dict(pos_df=pos_df, hard_neg_df=hard_neg_df, item_pool=item_pool, popularity=popularity, user_history=user_history)


In [8]:
result = build_recall_tables(df,"101","205","click")

In [9]:
result["pos_df"].head()

,click,purchase,101,121,122,124,125,126,127,128,...,127_14,150_14,D109_14,D110_14,D127_14,D150_14,D508,D509,D702,D853
0,1,0,114955,1,5,1,2,0,3,1,...,8343,71,0.1323,0.27700,0.0856,0.5180,0.4307,0.0000,0.0,0.1306
1,1,0,0,0,0,0,0,0,0,0,...,31070,13210,0.3179,0.09235,0.0856,0.2477,0.5347,0.0000,0.0,0.3188
2,1,0,129852,22,8,1,5,3,1,1,...,104,38,0.1670,0.09235,0.0856,0.2900,0.1511,0.0000,0.0,0.1943
3,1,0,207656,0,0,0,0,0,0,0,...,14834,36,0.1670,0.09235,0.0856,0.3490,0.3706,0.1222,0.0,0.1958
4,1,0,98161,8,1,1,1,1,1,1,...,1850,185,0.1670,0.09235,0.2212,0.5566,0.4932,0.1222,0.0,0.2688


In [10]:
print(len(result["pos_df"]))

836165


In [11]:
print(len(result["hard_neg_df"]))

20672142


In [12]:
print(len(result['item_pool']))
print(min(result['item_pool']))
print(max(result['item_pool']))

522499
0
538375


In [13]:
len(result['popularity'])

232452

In [14]:
len(result['user_history'])

93497

In [15]:
class RecallPosDataset(Dataset):
    """
    召回正样本 Dataset。
    每条样本只返回一个正样本行（user_id + item_id + 可选特征列）。
    负样本构建推迟到 RecallCollator，以支持 in-batch 策略。
    """

    def __init__(
        self,
        pos_df: pd.DataFrame,
        feature_cols: List[str],
        user_col: str = 'user_id',
        item_col: str = 'item_id',
    ):
        self.pos_df = pos_df.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.user_col = user_col
        self.item_col = item_col

    def __len__(self):
        return len(self.pos_df)

    def __getitem__(self, idx) -> Dict[str, torch.Tensor]:
        row = self.pos_df.iloc[idx]
        sample = {
            'user_id': torch.tensor(row[self.user_col], dtype=torch.long),
            'item_id': torch.tensor(row[self.item_col], dtype=torch.long),
        }
        for col in self.feature_cols:
            val = row[col]
            if isinstance(val, (list, np.ndarray)):
                sample[col] = torch.tensor(val, dtype=torch.float)
            elif isinstance(val, (int, np.integer)):
                sample[col] = torch.tensor(val, dtype=torch.long)
            else:
                sample[col] = torch.tensor(float(val), dtype=torch.float)
        return sample


In [ ]:
'''
pos tables
neg tables
items

1.get user feature , item features from pos table ,  item features from items
2.user features[batch size, user feature size]
3.pos item features[batch size, 1, item feature size]
4.neg item features[batch size, number of negative samples, item feature size]
5.in batch neg item features[batch size, batch size - 1, item feature size]
6.concat [batch size, 1 + number of negative samples + batch size - 1, item feature size]
7.compute scores [batch size,1 + number of negative samples + batch size - 1]
8.compute softmax scores
'''

In [29]:
user_sparse_features = [
    SparseFeature(name=name,vocab_size=vocabulary_size[name],embed_dim=32) for name in USER_SPARSE
]
user_dense_features = [
    DenseFeature(name=name, dim=1) for name in USER_DENSE
]

user_sequence_features = [ ]

item_sparse_features = [
    SparseFeature(name=name,vocab_size=vocabulary_size[name],embed_dim=32) for name in ITEM_SPARSE
]

item_dense_features = [
    DenseFeature(name=name, dim=1) for name in ITEM_DENSE
]

item_sequence_features = []

In [28]:
class RecallCollator:
    """
    构建完整召回 batch，返回完整 item 特征（走物品塔）。

    输出结构：
        user_features        : Dict[str, Tensor(B, ...)]        用户特征
        pos_item_features    : Dict[str, Tensor(B, ...)]        正样本物品特征
        neg_item_features    : Dict[str, Tensor(B, K, ...)]     简单负样本物品特征
    其中 in-batch 负样本在 TwoTowerModel 内部由 pos_item_features 构建。
    """

    def __init__(
        self,
        items_df: pd.DataFrame,
        item_sparse_features: List[SparseFeature],
        item_dense_features: List[DenseFeature],
        item_sequence_features: List[SequenceFeature],
        item_pool: np.ndarray,
        user_history: Dict,
        num_easy_neg: int = 4,
        user_col: str = 'user_id',
        item_col: str = 'item_id',
    ):
        # 以 item_id 为索引，方便 O(1) 查找
        self.items_df = items_df.set_index(item_col) if item_col in items_df.columns else items_df
        self.item_sparse_features = item_sparse_features
        self.item_dense_features = item_dense_features
        self.item_sequence_features = item_sequence_features
        self.item_pool = item_pool
        self.user_history = user_history
        self.num_easy_neg = num_easy_neg
        self.user_col = user_col
        self.item_col = item_col

    def _sample_easy_negs(self, user_id: int, n: int) -> List[int]:
        history = self.user_history.get(int(user_id), set())
        mask = np.array([i not in history for i in self.item_pool], dtype=bool)
        candidates = self.item_pool[mask]
        if len(candidates) == 0:
            candidates = self.item_pool
        return np.random.choice(candidates, size=n, replace=len(candidates) < n).tolist()

    def _extract_item_features(self, item_id: int) -> Dict[str, torch.Tensor]:
        """从 items_df 中提取单个 item 的完整特征。"""
        if item_id in self.items_df.index:
            row = self.items_df.loc[item_id]
        else:
            row = None

        features = {}
        for feat in self.item_sparse_features:
            val = int(row[feat.name]) if row is not None else 0
            features[feat.name] = torch.tensor(val, dtype=torch.long)
        for feat in self.item_dense_features:
            if row is not None:
                raw = row[feat.name]
                val = eval(raw) if isinstance(raw, str) else raw
            else:
                val = [0.0] * feat.dim if feat.dim > 1 else 0.0
            features[feat.name] = torch.tensor(val, dtype=torch.float)
        for feat in self.item_sequence_features:
            if row is not None:
                seq = row[feat.name]
                if isinstance(seq, str):
                    seq = eval(seq)
                seq = list(seq)
            else:
                seq = []
            seq_len = len(seq)
            if seq_len < feat.max_len:
                seq = seq + [0] * (feat.max_len - seq_len)
            else:
                seq = seq[:feat.max_len]
            features[feat.name] = torch.tensor(seq, dtype=torch.long)
        return features

    def _stack_item_features(self, feat_list: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        """将多个 item 特征字典 stack 成一个 batch 维度的字典。"""
        return {k: torch.stack([f[k] for f in feat_list]) for k in feat_list[0]}

    def __call__(self, samples: List[Dict[str, torch.Tensor]]) -> Tuple[
        Dict[str, torch.Tensor],
        Dict[str, torch.Tensor],
        Dict[str, torch.Tensor],
    ]:
        # 分离 user / item 字段
        item_keys = {feat.name for feat in self.item_sparse_features} \
                  | {feat.name for feat in self.item_dense_features} \
                  | {feat.name for feat in self.item_sequence_features}
        user_keys = set(samples[0].keys()) - item_keys - {self.item_col}

        user_features = {k: torch.stack([s[k] for s in samples]) for k in user_keys}
        user_ids = user_features[self.user_col]  # (B,)
        pos_item_ids = torch.stack([s[self.item_col] for s in samples])  # (B,)

        # 正样本特征 (B, ...)
        pos_feats_list = [self._extract_item_features(int(iid)) for iid in pos_item_ids]
        pos_item_features = self._stack_item_features(pos_feats_list)

        # 简单负样本特征 (B, K, ...)
        neg_feats_per_user = []
        for uid in user_ids:
            neg_ids = self._sample_easy_negs(int(uid), self.num_easy_neg)
            neg_feats = [self._extract_item_features(nid) for nid in neg_ids]
            neg_feats_per_user.append(self._stack_item_features(neg_feats))  # Dict[str, (K,...)]

        # stack across batch: Dict[str, (B, K, ...)]
        neg_item_features = {
            k: torch.stack([neg_feats_per_user[i][k] for i in range(len(samples))])
            for k in neg_feats_per_user[0]
        }

        return user_features, pos_item_features, neg_item_features

In [19]:
dataset = RecallPosDataset(result["pos_df"],["121", "122", "124", "125", "126", "127", "128", "129", "206", "207", "210", "216"], "101", "205")

In [30]:
collator = RecallCollator(df,item_sparse_features,item_dense_features,item_sequence_features,result['item_pool'],result['user_history'],128,'user_id','item_id')

In [31]:
print(len(dataset))
print(dataset[0])

836165
{'user_id': tensor(114955), 'item_id': tensor(8720), '121': tensor(1.), '122': tensor(5.), '124': tensor(1.), '125': tensor(2.), '126': tensor(0.), '127': tensor(3.), '128': tensor(1.), '129': tensor(0.), '206': tensor(90.), '207': tensor(8485.), '210': tensor(1663.), '216': tensor(0.)}


In [32]:
dataset[1]

{'user_id': tensor(0),
 'item_id': tensor(46250),
 '121': tensor(0.),
 '122': tensor(0.),
 '124': tensor(0.),
 '125': tensor(0.),
 '126': tensor(0.),
 '127': tensor(0.),
 '128': tensor(0.),
 '129': tensor(0.),
 '206': tensor(1323.),
 '207': tensor(15114.),
 '210': tensor(32695.),
 '216': tensor(21009.)}

In [33]:
loader = DataLoader(
            dataset, batch_size=128,
            shuffle=True, num_workers=4,
            collate_fn=collator, pin_memory=True,
        )

In [34]:
batch = next(iter(loader))

In [38]:
batch[0].keys()

dict_keys(['122', 'user_id', '129', '126', '124', '125', '121', '128', '127'])

In [39]:
batch[1].keys()

dict_keys(['206', '207', '210', '216'])

In [42]:
batch[1]['206'].shape

torch.Size([128])

In [40]:
batch[2].keys()

dict_keys(['206', '207', '210', '216'])

In [43]:
batch[2]['206'].shape

torch.Size([128, 128])

In [53]:
batch['easy_neg_ids'].shape

torch.Size([128, 128])

In [51]:
batch['121'].shape

torch.Size([128])

In [49]:
batch['inbatch_correction'].shape

torch.Size([128, 128])

In [43]:
batch['122']

tensor([ 6.,  8.,  8.,  1.,  1.,  6.,  9.,  8.,  1.,  6.,  1.,  4.,  6.,  1.,
         3., 12.,  2.,  6.,  2.,  6.,  3.,  1.,  6.,  5.,  8.,  4., 12.,  1.,
         6.,  5.,  3.,  2.,  0., 11., 11.,  5.,  8.,  2., 11.,  1.,  5.,  2.,
         5.,  1.,  6.,  6.,  0., 12.,  6.,  8.,  6.,  1.,  0.,  8.,  8.,  1.,
         6.,  6.,  1.,  8.,  1.,  3.,  2.,  7.,  9.,  1., 11.,  6.,  6.,  9.,
         7.,  3.,  6.,  7.,  2.,  2.,  1.,  1.,  6.,  8.,  1.,  6.,  5.,  0.,
         1.,  5.,  6.,  5.,  0.,  1.,  1.,  1.,  6.,  5.,  1.,  6.,  6.,  1.,
        10.,  8.,  5.,  1.,  1.,  0.,  0.,  1.,  7.,  1.,  5.,  8.,  8.,  0.,
         0.,  1.,  7.,  1., 11.,  0.,  6.,  7.,  6.,  5.,  1.,  0.,  8.,  2.,
         2.,  8.])

In [ ]:
class TwoTowerDataModule(pl.LightningDataModule):
    def __init__(
        self,
        raw_df: pd.DataFrame,
        feature_cols: List[str],
        num_easy_neg: int = 4, # 每条样本的简单负样本数
        val_ratio: float = 0.1, # 验证集比例
        batch_size: int = 256,
        num_workers: int = 4,
        user_col: str = 'user_id',
        item_col: str = 'item_id',
        label_col: str = 'click',
        hard_neg_col: Optional[str] = None,   # 困难负样本列（可选）
    ):
        super().__init__()
        self.feature_cols = feature_cols
        self.num_easy_neg = num_easy_neg
        self.val_ratio = val_ratio
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.user_col = user_col
        self.item_col = item_col

        # 预处理：拆分正负样本、统计热度、构建用户历史
        tables = build_recall_tables(raw_df, user_col, item_col, label_col, hard_neg_col)
        self.pos_df = tables['pos_df']
        self.hard_neg_df = tables['hard_neg_df']
        self.item_pool = tables['item_pool']
        self.popularity = tables['popularity']
        self.user_history = tables['user_history']

    def setup(self, stage: Optional[str] = None):
        """随机划分 train / val 并初始化 Dataset 和 Collator。"""
        n = len(self.pos_df)
        n_val = max(1, int(n * self.val_ratio))
        idx = np.random.permutation(n)
        val_idx, train_idx = idx[:n_val], idx[n_val:]

        self.train_dataset = RecallPosDataset(
            self.pos_df.iloc[train_idx], self.feature_cols, self.user_col, self.item_col
        )
        self.val_dataset = RecallPosDataset(
            self.pos_df.iloc[val_idx], self.feature_cols, self.user_col, self.item_col
        )

        # Collator 负责在 batch 维度构建负样本和热度校正矩阵
        self.collator = RecallCollator(
            item_pool=self.item_pool,
            popularity=self.popularity,
            user_history=self.user_history,
            num_easy_neg=self.num_easy_neg,
            user_col=self.user_col,
            item_col=self.item_col,
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset, batch_size=self.batch_size,
            shuffle=True, num_workers=self.num_workers,
            collate_fn=self.collator, pin_memory=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset, batch_size=self.batch_size,
            shuffle=False, num_workers=self.num_workers,
            collate_fn=self.collator, pin_memory=True,
        )

In [55]:
batch.keys()

dict_keys(['user_id', 'item_id', '121', '122', '124', '125', '126', '127', '128', '129', '206', '207', '210', '216', 'easy_neg_ids', 'inbatch_correction'])